## Script to plot all relevant figures 

In [417]:
# import packages
import pandas as pd
import os
import requests

from cost_flow_map_functions import *
from LNG_external_imports_functions import *
from year_difference_functions import *

In [418]:
years = [2021, 2024, 2035]
save_output = True 
output_path = os.path.join('..', '..', '02_plots')

In [731]:
year = 2035
year_comparison = 2035

scenario_considered = "2035_AP"
# Choice between :
# '2019', '2021', '2024', '2024_inv', '2024_plus_no_QA', '2024_plus_NO_reduced', '2024_plus_no_USA', '2024_with_RU', '2035_SP', '2035_AP'

paper_folder = '02_paper_ESR' # Change to '02_paper_ESR' for ESR paper outputs, esle '01_paper_IAEE' for IAEE outputs
paper = "ESR_2026" # Change to "IAEE_2025" for IAEE outputs

In [732]:
country_map = {
    'AF': 'Africa', 
    'EG': 'Egypt',
    'QA': 'Qatar',
    'TT': 'Trinidad & Tobago',
    'USA': 'United States'
}

### Load all useful result files

In [733]:
data_path = os.path.join('..', '..', '01_data')

In [734]:
input_excluded_pipelines = os.path.join(data_path, "01_input_data", "01_raw", "01_Russian_War_Case", "information_pipelines_exclude_from_plots.xlsx")

In [735]:
res_2021 = os.path.join(data_path, "02_output_data", "02_unidirectional_results", "01_paper_IAEE", "02_prepared_results", "output_2021_prepared.xlsx")
res_2024 = os.path.join(data_path, "02_output_data", "02_unidirectional_results", "01_paper_IAEE", "02_prepared_results", "output_2024_prepared.xlsx")

In [736]:
baseline_file_2021 = "outputs_IAEE_2025_run_2021"
baseline_file_2024 = "outputs_IAEE_2025_run_2024"
baseline_file_2035 = "outputs_IAEE_2025_run_2035_SP"

baseline_path_2021 = os.path.join(data_path, "02_output_data", "02_unidirectional_results", "01_paper_IAEE", "01_raw_results", baseline_file_2021+ ".xlsx")
baseline_path_2024 = os.path.join(data_path, "02_output_data", "02_unidirectional_results", "01_paper_IAEE", "01_raw_results", baseline_file_2024+ ".xlsx")
baseline_path_2035 = os.path.join(data_path, "02_output_data", "02_unidirectional_results", "01_paper_IAEE", "01_raw_results", baseline_file_2035+ ".xlsx")

In [737]:
df_output_2021 = pd.read_excel(res_2021,sheet_name='LNG_Sources')
df_output_2024 = pd.read_excel(res_2024, sheet_name='LNG_Sources')
flow_column = 'Flow (GWh).1'

### Pie chart showing different importers per year

In [738]:
plotly_pie_charts(df_output_2021, df_output_2024, flow_column, 2021, 2024)

### Plots showing baseline plot per year

In [739]:
input_LNG_file = os.path.join(data_path, "01_input_data", "01_raw", "01_Russian_War_Case", "LNG_locations.xlsx")
filtered_ports = filter_lng_ports_by_year(input_LNG_file, years)

##### Fix the ports location for better visualization

In [740]:
filtered_ports.loc[filtered_ports['Name of \ninstallation'] == 'Kollsnes 1', 'Longitude'] += 0.4
filtered_ports.loc[filtered_ports['Name of \ninstallation'] == 'Kollsnes 2', 'Longitude'] += 0.4

filtered_ports['Name of \ninstallation'] = filtered_ports['Name of \ninstallation'].str.strip()
filtered_ports.loc[filtered_ports['Name of \ninstallation'] == 'Mukran FSRU Energos Power', 'Latitude'] -= 0.5
filtered_ports = filtered_ports[filtered_ports['Name of \ninstallation'] != 'Mukran FSRU Neptune – 2nd']

filtered_ports.loc[filtered_ports['Name of \ninstallation'] == 'Mag Mell FSRU', 'Latitude'] += 0.5

In [741]:
countries_with_terminals = get_countries_with_terminals(filtered_ports)
countries_iso2 = set(filter(None, [country_name_to_code(name) for name in countries_with_terminals]))
filtered_LNG_IMPORT_COORDS = {country: coords for country, coords in LNG_IMPORT_COORDS.items() if country in countries_iso2}

##### Process each file and plot

In [742]:
df_ports_2021 = identify_terminal_status(filtered_ports[filtered_ports['Model Year'] <= 2021], 2021)
df_2021 = process_file(baseline_path_2021)
pipelines_2021 = df_2021[(df_2021['FromType'] == '-') & (df_2021['ToType'] == '-')]
pipeline_status_2021 = {edge: 'included' for edge in pipelines_2021['Edge']}

In [743]:
df_ports_2024 = identify_terminal_status(filtered_ports[filtered_ports['Model Year'] <= 2024], 2024)
df_2024 = process_file(baseline_path_2024)
pipelines_2024 = df_2024[(df_2024['FromType'] == '-') & (df_2024['ToType'] == '-')]
pipeline_status_2024 = scenario_pipeline_exclusions(input_excluded_pipelines, pipelines_2024['Edge'].dropna().unique())

In [744]:
df_ports_2035 = identify_terminal_status(filtered_ports, 2035)
df_2035 = process_file(baseline_path_2035)
pipelines_2035 = df_2035[(df_2035['FromType'] == '-') & (df_2035['ToType'] == '-')]
pipeline_status_2035 = {edge: 'included' for edge in pipelines_2035['Edge']}

In [745]:
output_file = os.path.join(output_path, "base_map_2x2.png")

In [746]:
plot_three_years_2x2(df_ports_2021, pipelines_2021, pipeline_status_2021,
                         df_ports_2024, pipelines_2024, pipeline_status_2024,
                         df_ports_2035, pipelines_2035, pipeline_status_2035,
                         output_file, save_output)

### Plot flow maps 

In [747]:
file_name = f"outputs_ESR_2026_run_{scenario_considered}"

raw_path = os.path.join(data_path, "02_output_data", "02_unidirectional_results", paper_folder, "01_raw_results", file_name+ ".xlsx")
result_path = os.path.join(data_path, "02_output_data", "02_unidirectional_results", paper_folder, "02_prepared_results", file_name+"_utilization_share.xlsx")
output_path_flow = os.path.join(output_path, "Flow_Results", file_name)

In [748]:
df = process_file(raw_path)

##### Compute flows and capacity shares per pipeline

In [749]:
df_with_imports = df[(df['FromType'] == 'LNG_import')]

In [750]:
utilization_df = utilization_df = (df_with_imports[["From", "Flow", "Capacity_tot", "Share"]]
    .rename(columns={"From": "Country"})
    .sort_values(by="Country", ascending=True))

In [751]:
total_flow = utilization_df["Flow"].sum()
total_capacity = utilization_df["Capacity_tot"].sum()

In [752]:
total_flow_EU = utilization_df[utilization_df.Country.apply(eu_countries)].Flow.sum()
total_capacity_EU = utilization_df[utilization_df.Country.apply(eu_countries)].Capacity_tot.sum()

In [753]:
total_share = total_flow / total_capacity if total_capacity != 0 else 0
total_share_EU = total_flow_EU / total_capacity_EU if total_capacity_EU != 0 else 0

In [754]:
summary_row = pd.DataFrame({
    "Country": ["Total"],
    "Flow": [total_flow],
    "Capacity_tot": [total_capacity],
    "Share": [total_share]
})

In [755]:
summary_row_EU = pd.DataFrame({
    "Country": ["Total_EU"],
    "Flow": [total_flow_EU],
    "Capacity_tot": [total_capacity_EU],
    "Share": [total_share_EU]
})

In [756]:
utilization_summary = pd.concat([utilization_df, summary_row, summary_row_EU], ignore_index=True)

In [757]:
bar_fig = plot_bar_chart(utilization_summary)

##### Show flow maps accounting for excluded pipelines

In [758]:
df_final = process_pipelines(df, file_name, input_excluded_pipelines)

In [759]:
fig = plot_flow_map(df_final, filtered_ports, df_with_imports, filtered_LNG_IMPORT_COORDS)

In [760]:
if save_output: 
    os.makedirs(os.path.dirname(result_path), exist_ok=True)
    fig.write_image(output_path_flow + ".png", width=900, height=650, scale=2)
    # utilization_summary.to_excel(result_path, index=False)
    bar_fig.write_image(output_path_flow + '_bar_chart.png', width=1135, height=800, scale=2)

### Show maps for cost 

In [761]:
url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_UKR_0.json"
json_file_path = os.path.join("gadm41_UKR_0.json")
response = requests.get(url)
with open(json_file_path, "wb") as f:
    f.write(response.content)

print("JSON File downloaded successfully")

JSON File downloaded successfully


In [762]:
# To change with new ESR paper folder 
results_folder = os.path.join(data_path, "02_output_data", "02_unidirectional_results", paper_folder, "02_prepared_results")

In [763]:
full_name = f"costs_shares_{paper}_run_{scenario_considered}"

In [764]:
input_file_cost = os.path.join(results_folder, f"costs_shares_{paper}_run_{scenario_considered}.xlsx")

In [765]:
plot_cost_map(input_file_cost, scenario_considered, output_path, json_file_path, save_output)

In [766]:
if paper == "ESR_2026":
    file_cost_difference_2019 = os.path.join(results_folder, "cost_shares_differences_to_2019.xlsx")
file_cost_difference_2021 = os.path.join(results_folder, "cost_shares_differences_to_2021.xlsx")
file_cost_difference_2024 = os.path.join(results_folder, "cost_shares_differences_to_2024.xlsx")
file_cost_difference_2035 = os.path.join(results_folder, "cost_shares_differences_to_2035_SP.xlsx")

In [767]:
colorbar_zero_2021 = 0.32
colorbar_zero_2024 = 0.65
colorbar_zero_2035 = 0.77

In [768]:
global_min_2019 = -13700
global_max_2019 = 11875
global_min_2021 = -13700
global_max_2021 = 11875
global_min_2024 = -13700
global_max_2024 = 10500
global_min_2035 = -13700
global_max_2035 = 12000

In [769]:
if year_comparison == 2019:
    plot_cost_difference(file_cost_difference_2019, full_name, scenario_considered, output_path, global_min_2019, global_max_2019, json_file_path, save_output)
elif year_comparison == 2021:
    plot_cost_difference(file_cost_difference_2021, full_name, scenario_considered, output_path, global_min_2021, global_max_2021, json_file_path, save_output)
elif year_comparison == 2024:
    plot_cost_difference(file_cost_difference_2024, full_name, scenario_considered, output_path, global_min_2024, global_max_2024, json_file_path, save_output)
elif year_comparison == 2035:
    plot_cost_difference(file_cost_difference_2035, full_name, scenario_considered, output_path, global_min_2035, global_max_2035, json_file_path, save_output)

KeyError: 'costs_shares_ESR_2026_run_2035_AP'

### Plot emission differences with bars

In [ ]:
data_path_emissions = os.path.join(data_path, "02_output_data", "02_unidirectional_results", f"{paper_folder}", "02_prepared_results")
input_file_emissions= os.path.join(data_path_emissions, "emission_differences.xlsx")
input_file_factors = os.path.join(data_path_emissions, "emission_factors.xlsx")

order = [# '2019',
         'REF', 'REX', 'REX - US variation', 'REX - Qatari variation',
         'REX - Norwegian variation', 'REX - Russian variation', 'AR',
         'PLE - SP variation', 'PLE - AP variation']

In [ ]:
emissions_diff = pd.read_excel(input_file_emissions, sheet_name="Emissions_Europe")

In [ ]:
factor_eu = pd.read_excel(input_file_factors, sheet_name="Emission_Factor_EU")
factor_europe = pd.read_excel(input_file_factors, sheet_name="Emission_Factor_Europe")

In [ ]:
factor_eu['Scenario'] = factor_eu['Scenario'].str.replace("ESR_2026_run_", "")
factor_europe['Scenario'] = factor_europe['Scenario'].str.replace("ESR_2026_run_", "")

factor_eu['Scenario'] = factor_eu['Scenario'].replace({'2024_plus_no_QA':'2024_no_QA', '2024_plus_no_USA':'2024_no_USA', '2024_plus_NO_reduced':'2024_NO_red'})
factor_europe['Scenario'] = factor_europe['Scenario'].replace({'2024_plus_no_QA':'2024_no_QA', '2024_plus_no_USA':'2024_no_USA', '2024_plus_NO_reduced':'2024_NO_red'})

In [ ]:
emissions_diff = emissions_diff.merge(factor_eu[['Scenario','tCO2e/GWh in the EU']], on='Scenario',how='left')
emissions_diff.rename(columns={'tCO2e/GWh in the EU':'Emission_Factor_EU'},inplace=True)

emissions_diff = emissions_diff.merge(factor_europe[['Scenario','tCO2e/GWh in Europe']],on='Scenario',how='left')
emissions_diff.rename(columns={'tCO2e/GWh in Europe':'Emission_Factor_Europe'}, inplace=True)

In [ ]:
emissions_diff.loc[emissions_diff['Scenario'] == '2021', 'Scenario'] = 'REF'
emissions_diff.loc[emissions_diff['Scenario'] == '2024', 'Scenario'] = 'REX'
emissions_diff.loc[emissions_diff['Scenario'] == '2024_no_USA', 'Scenario'] = 'REX - US variation'
emissions_diff.loc[emissions_diff['Scenario'] == '2024_no_QA', 'Scenario'] = 'REX - Qatari variation'
emissions_diff.loc[emissions_diff['Scenario'] == '2024_NO_red', 'Scenario'] = 'REX - Norwegian variation'
emissions_diff.loc[emissions_diff['Scenario'] == '2024_with_RU', 'Scenario'] = 'REX - Russian variation'
emissions_diff.loc[emissions_diff['Scenario'] == '2024_inv', 'Scenario'] = 'AR'
emissions_diff.loc[emissions_diff['Scenario'] == '2035_SP', 'Scenario'] = 'PLE - SP variation'
emissions_diff.loc[emissions_diff['Scenario'] == '2035_AP', 'Scenario'] = 'PLE - AP variation'
#emissions_diff.sort_values(by='Scenario', inplace=True)

In [ ]:
emissions_diff['Scenario'] = pd.Categorical(emissions_diff['Scenario'], categories=order, ordered=True)

In [ ]:
plot_emission_difference_factors(emissions_diff, "2021", 'REF', output_path, save_output)